In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
branch_table = dbutils.widgets.get("branch_table")
office_table = dbutils.widgets.get("office_table")
clientepisodefsall_table = dbutils.widgets.get("clientepisodefsall_table")
payorsources_table = dbutils.widgets.get("payorsources_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW customfile_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS INT) AS FacilityCode,
  AcctNbr AS AcctNbr,
  Company AS Company,
  Practice AS Practice,
  BayadaRegion AS BayadaRegion,
  BayadaDivision AS BayadaDivision,
  BayadaOfficeAbbreviation AS BayadaOfficeAbbreviation,
  NULL AS CompanyTaxID,
  ReimbursementTeam AS ReimbursementTeam,
  PayerSourceNumber AS PayerSourceNumber,
  NULL AS PayerSourceProgramName,
  NULL AS BillingPeriodOrFrequency,
  NULL AS EVVRequirements,
  NULL AS EVVAggregator,
  NULL AS EVVCodesinScope,
  NULL AS COBPrimaryInsurance, 
  NULL AS ReferralID,
  NULL AS PrimarySubscriberIDNumber,
  NULL AS COBStatus,
  NULL AS COBType,
  NULL AS COBEffDate,
  NULL AS COBLevel,
  NULL AS COBDenialReason,
  NULL AS COBSplitDecision,
  NULL AS LimitedBenefit,
  NULL AS BenefitDate,
  NULL AS BillHoldReason,
  NULL AS BillHoldDate,
  CAST(SourceSystemKey AS INT) AS SourceSystemKey
FROM (
  WITH 
  customfile_cte AS (
    SELECT 
    CAST('{fetch_date}' AS DATE) AS ReportingDate,
    CASE 
        WHEN b.branch_code RLIKE '[A-Za-z]' THEN ofc.OfficeNumber 
        ELSE b.branch_code 
    END AS FacilityCode,
    bi_h.i_id as AcctNbr,
    "BAYADA Home Health Care, Inc." AS Company,
    ofc.Practice AS Practice,
    ofc.Region AS BayadaRegion,
    ofc.Division AS BayadaDivision,
    ofc.OfficeAbbreviation AS BayadaOfficeAbbreviation,
    ofc.ReimbursementOfficeNumber AS ReimbursementTeam,
    ps.ps_id AS PayerSourceNumber, -- needs to be discovered
    '6' AS SourceSystemKey
    FROM {source_table} bi_h
    JOIN {branch_table} b 
        ON bi_h.i_branchcode = b.branch_code
    LEFT JOIN {office_table} ofc
        ON ofc.OfficeAbbreviation = b.branch_code
    JOIN {clientepisodefsall_table} cefs
        ON cefs.cefs_id = bi_h.i_cefsid
    LEFT JOIN {payorsources_table} ps
        ON ps.ps_id = cefs.cefs_psid
    WHERE bi_h.i_Balance <> 0
  ),
  customfile_clean AS (
    SELECT *,
    row_number() OVER (PARTITION BY AcctNbr ORDER BY AcctNbr ) AS rn
    FROM customfile_cte
  )
  SELECT 
    ReportingDate, 
    FacilityCode, 
    AcctNbr, 
    Company, 
    Practice, 
    BayadaRegion, 
    BayadaDivision, 
    BayadaOfficeAbbreviation, 
    ReimbursementTeam, 
    PayerSourceNumber, 
    SourceSystemKey
  FROM customfile_clean
  WHERE rn=1
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING customfile_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 6

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.Company = src.Company,
    tgt.Practice = src.Practice,
    tgt.BayadaRegion = src.BayadaRegion,
    tgt.BayadaDivision = src.BayadaDivision,
    tgt.BayadaOfficeAbbreviation = src.BayadaOfficeAbbreviation,
    tgt.CompanyTaxID = src.CompanyTaxID,
    tgt.ReimbursementTeam = src.ReimbursementTeam,
    tgt.PayerSourceNumber = src.PayerSourceNumber,
    tgt.PayerSourceProgramName = src.PayerSourceProgramName,
    tgt.BillingPeriodOrFrequency = src.BillingPeriodOrFrequency,
    tgt.EVVRequirements = src.EVVRequirements,
    tgt.EVVAggregator = src.EVVAggregator,
    tgt.EVVCodesinScope = src.EVVCodesinScope,
    tgt.COBPrimaryInsurance = src.COBPrimaryInsurance,
    tgt.ReferralID = src.ReferralID,
    tgt.PrimarySubscriberIDNumber = src.PrimarySubscriberIDNumber,
    tgt.COBStatus = src.COBStatus,
    tgt.COBType = src.COBType,
    tgt.COBEffDate = src.COBEffDate,
    tgt.COBLevel = src.COBLevel,
    tgt.COBDenialReason = src.COBDenialReason,
    tgt.COBSplitDecision = src.COBSplitDecision,
    tgt.LimitedBenefit = src.LimitedBenefit,
    tgt.BenefitDate = src.BenefitDate,
    tgt.BillHoldReason = src.BillHoldReason,
    tgt.BillHoldDate = src.BillHoldDate,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    Company,
    Practice,
    BayadaRegion,
    BayadaDivision,
    BayadaOfficeAbbreviation,
    CompanyTaxID,
    ReimbursementTeam,
    PayerSourceNumber,
    PayerSourceProgramName,
    BillingPeriodOrFrequency,
    EVVRequirements,
    EVVAggregator,
    EVVCodesinScope,
    COBPrimaryInsurance,
    ReferralID,
    PrimarySubscriberIDNumber,
    COBStatus,
    COBType,
    COBEffDate,
    COBLevel,
    COBDenialReason,
    COBSplitDecision,
    LimitedBenefit,
    BenefitDate,
    BillHoldReason,
    BillHoldDate,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.Company,
    src.Practice,
    src.BayadaRegion,
    src.BayadaDivision,
    src.BayadaOfficeAbbreviation,
    src.CompanyTaxID,
    src.ReimbursementTeam,
    src.PayerSourceNumber,
    src.PayerSourceProgramName,
    src.BillingPeriodOrFrequency,
    src.EVVRequirements,
    src.EVVAggregator,
    src.EVVCodesinScope,
    src.COBPrimaryInsurance,
    src.ReferralID,
    src.PrimarySubscriberIDNumber,
    src.COBStatus,
    src.COBType,
    src.COBEffDate,
    src.COBLevel,
    src.COBDenialReason,
    src.COBSplitDecision,
    src.LimitedBenefit,
    src.BenefitDate,
    src.BillHoldReason,
    src.BillHoldDate,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)